In [ ]:
import pandas as pd
import numpy as np
import ast

In [ ]:
movies = pd.read_csv('movies_metadata.csv')
keywords=pd.read_csv('keywords.csv')
credits = pd.read_csv(
    'credits.csv',
    low_memory=False
)

/tmp/ipykernel_1507/2466890255.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv('movies_metadata.csv')


In [ ]:
movies.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [ ]:
movies = movies[
    [
        'id',
        'title',
        'overview',
        'genres',
        'poster_path',
        'vote_average',
        'release_date'
    ]
]

In [ ]:
movies['id'] = movies['id'].astype(str)
credits['id'] = credits['id'].astype(str)
keywords['id'] = keywords['id'].astype(str)

In [ ]:
movies = movies.merge(keywords, on='id')
movies = movies.merge(credits, on='id')

In [ ]:
def convert(obj):

    L = []

    for i in ast.literal_eval(obj):
        L.append(i['name'])

    return L

In [ ]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
def convert_cast(obj):

    L = []

    counter = 0

    for i in ast.literal_eval(obj):

        if counter != 3:
            L.append(i['name'])
            counter += 1

        else:
            break

    return L

In [ ]:
movies['cast'] = movies['cast'].apply(convert_cast)

In [ ]:
def fetch_director(obj):

    L = []

    for i in ast.literal_eval(obj):

        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies['overview'] = movies['overview'].fillna('')

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [ ]:
movies['genres'] = movies['genres'].apply(
    lambda x: [i.replace(" ", "") for i in x]
)

movies['keywords'] = movies['keywords'].apply(
    lambda x: [i.replace(" ", "") for i in x]
)

movies['cast'] = movies['cast'].apply(
    lambda x: [i.replace(" ", "") for i in x]
)

movies['crew'] = movies['crew'].apply(
    lambda x: [i.replace(" ", "") for i in x]
)

In [ ]:
movies['tag'] = (
    movies['overview']
    + movies['genres']
    + movies['keywords']
    + movies['cast']
    + movies['crew']
)

In [ ]:
new_data = movies[
    [
        'id',
        'title',
        'tag',
        'overview',
        'poster_path',
        'vote_average',
        'release_date'
    ]
].copy()

In [ ]:
new_data['tag'] = new_data['tag'].apply(
    lambda x: " ".join(x)
)

In [ ]:
new_data['tag'] = new_data['tag'].fillna('')
new_data['title'] = new_data['title'].fillna('')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import pickle

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words='english'
)

vectors = tfidf.fit_transform(new_data['tag'])

In [ ]:
model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=11
)

model.fit(vectors)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=11)

In [ ]:
def recommend(movie):

    movie = movie.lower()

    matches = new_data[
        new_data['title']
        .fillna('')
        .str.lower()
        .str.contains(movie)
    ]

    if matches.empty:
        return []

    movie_index = matches.index[0]

    distances, indices = model.kneighbors(
        vectors[movie_index],
        n_neighbors=11
    )

    recommended_movies = []

    for i in indices[0][1:]:

        movie_data = movies.iloc[i]

        recommended_movies.append({
            "id": int(movie_data.id),

            "title": movie_data.title,

            "overview": (
                " ".join(movie_data.overview)
                if isinstance(movie_data.overview, list)
                else movie_data.overview
            ),

            "poster": (
                "https://image.tmdb.org/t/p/w500"
                + str(movie_data.poster_path)
                if pd.notna(movie_data.poster_path)
                else None
            ),

            "rating": (
                float(movie_data.vote_average)
                if pd.notna(movie_data.vote_average)
                else None
            ),

            "release_date": movie_data.release_date,

            "genres": (
                ", ".join(movie_data.genres)
                if isinstance(movie_data.genres, list)
                else movie_data.genres
            )
        })

    return recommended_movies

In [ ]:
recommend("Interstellar")

[{'id': 18221,
  'title': 'Space Station 3D',
  'overview': "Some 220 miles above Earth lies the International Space Station, a one-of-a-kind outer space laboratory that 16 nations came together to build. Get a behind-the-scenes look at the making of this extraordinary structure in this spectacular IMAX film. Viewers will blast off from Florida's Kennedy Space Center and the Baikonur Cosmodrome in Russia for this incredible journey -- IMAX's first-ever space film. Tom Cruise narrates.",
  'poster': 'https://image.tmdb.org/t/p/w500/6RBqmI2tux6lx0LLojPCQcUL4lq.jpg',
  'rating': 6.1,
  'release_date': '2002-04-17',
  'genres': 'Documentary'},
 {'id': 29570,
  'title': 'Journey to the Seventh Planet',
  'overview': 'A space expedition to Uranus is menaced by a giant brain that can make illusions come true.',
  'poster': 'https://image.tmdb.org/t/p/w500/teVKcwWifphk2jDzFEuVMJlBiLz.jpg',
  'rating': 4.4,
  'release_date': '1962-03-10',
  'genres': 'Adventure, Fantasy, Horror, ScienceFiction'

In [ ]:
import pickle

pickle.dump(
    movies,
    open('movies.pkl', 'wb')
)
pickle.dump(
    new_data,
    open('new_data.pkl', 'wb')
)
pickle.dump(
    model,
    open('model.pkl', 'wb')
)
pickle.dump(
    tfidf,
    open('tfidf.pkl', 'wb')
)
pickle.dump(
    vectors,
    open('vectors.pkl', 'wb')
)

In [ ]:
moviesfull = pd.read_csv('movies_metadata.csv')

/tmp/ipykernel_1507/2244205811.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  moviesfull = pd.read_csv('movies_metadata.csv')


In [ ]:
movies.columns

Index(['id', 'title', 'overview', 'genres', 'poster_path', 'vote_average',
       'release_date', 'keywords', 'cast', 'crew', 'tag'],
      dtype='object')

In [ ]:
movies['runtime'] = moviesfull['runtime']

In [ ]:
pickle.dump(
    movies,
    open('movies.pkl', 'wb')
)